INTUIZIONE DELL'ALGORITMO DI BACKPROPAGATION

- Il viaggio a ritroso: come l'errore risale la rete dall'output verso l'input per correggere il tiro.
- Attribuzione della colpa: quantificare quanto ogni singolo peso sia responsabile dell'errore finale.
- Il Grafo Computazionale: l'infrastuttura logica che permette di automatizzare i calcoli complessi del gradiente, l'impalcatura che permette al computer di non impazzire tra miliardi di calcoli.

1) IL VIAGGIO A RITROSO invertire il senso di marcia

Il segnale viaggia in avanti (forward pass), tuttavia, dopo si guarda quanto si è andati lontano dalla verità e quello scarto è la loss function.
La back propagation è il momento in cui si analizza il tiro e si torna indietro mentalmente per capire come migliorare il tiro.

Si inverte il flusso, si parte dall'output e si mandano messaggi di correzione verso i layer precedenti.
E' un processo ricorsivo, per sapere come correggere il primo layer dobbiamo prima aver capito come correggere l'ultimo. Tutto questo parte dalla derivata della loss rispetto la predizione.

Ogni neurone quando riceve l'errore dal suo layer successivo lo moltiplica per il suo gradiente locale (sua personale sensibilità, quanto forte reagisco) I pesi delle connessioni agiscono poi come amplificatori o silenziatori, se un peso è molto grande, quel neurone si prenderà una fetta di colpa maggiore. E' un sistema meritocratico al contrario, più influenzi il risultato più verrai corretto se il risultato è sbagliato.
Tutto questo tramite la regola della catena (Chain Rule): ogni layer calcola solo una piccola parte della derivata totale, passando il risultato al layer successivo in  una catena di moltiplicazioni .
Invece di calcolare tutta la derivata mostruosa, la back propagation spezzetta il problema. Ogni layer risolve il suo piccolo pezzetto di derivata. Moltiplicando questi piccoli contributi si ottiene la variazione totale.
La collaborazione locale porta ad una soluzione globale.

Come decidiamo chi deve cambiare di più?

2) ATTRIBUZIONE DELLE RESPONSABILITA'

Non tutti i parametri (pesi) sono colpevoli allo stesso modo. Alcuni pesi influenzano il risultato finale più di altri. L'obiettivo della backpropagation è isolare l'impatto di ogni singolo valore numerico della Loss, vanno puniti specificatamente i pesi responsabili dell'errore.
Come si miura?
Si usa il concetto di pendenza, il gradiente ci dice in che direzione stiamo andando sulla collina dell'errore. Se il gradiente è positivo, aumentare il peso aumenta l'errore quindi dobbiamo diminuirlo.
Se il gradiente è negativo, facciamo l'oppposto. La magnitudo, quanto è grande quel numero, ci dice quanto agressivo deve essere il nostro aggiornameto.
Tutto questo deve essere calcolato per milioni di parametri contemporaneamente.
Anche se calcoliao le derivate singolarmente, il gradiente di un singolo layer contriene traccie in tutti i layer successivi creando una cascata di dipendenze che il gradiente deve navigare con precisione.
Per fae tutto questo abbiamo bisogno di una mappa, il grafo computazionale

3) GRAFO COMPUTAZIONALE

Il Grafo Computazionale è una rappresentazione visiva e logica delle operazioni matematiche. Ogni nodo è un'operazione e ogni arco rappresenta il flusso dei dati.
Per la backpropagation il grafo è fondamentale: permette al computer di tenere traccia di tutte le dipendenze necessarie per calcolare le derivate in modo automatico.
Una volta che si ha a disposizione la mappa (il grafo) di una rete neurale, è facilissimo percorrerla indietro, per capire dove arrivare l'errore.


In [1]:
import numpy as np

In [4]:
#1) Configurazione iniziale
x=1.5
y_true=0.8
w1=0.5 #Peso tra imput e hidden layer
w2=0.7 #Peso tra hidden layer e output layer

learning_rate=0.1

print(f"Inizio: input={x}, Target desiderato={y_true}")
print(f"Pesi iniziali: w1={w1}, w2={w2}")

Inizio: input=1.5, Target desiderato=0.8
Pesi iniziali: w1=0.5, w2=0.7


In [5]:
# FORWORD PASS

# Calcoliamo l'output della rete
h=x*w1 # Output del hidden layer (neurone nascosto), attivazione lineare
y_pred=h*w2 # Output del output layer (predizione)

#Calcolo della loss (Mean Squared Error per un singolo esempio)
loss=0.5*(y_pred-y_true)**2
print(f"Output predetto: {y_pred}, Loss: {loss}")

Output predetto: 0.5249999999999999, Loss: 0.037812500000000034


In [7]:
# BACKFORWARD PASS (Il cuore della lezione)

#1. Gradiente rispetto a W2 (ultimo layer)
# dloss/dw2 = dloss/dy_pred * dy_pred/dw2
dloss_dypred = y_pred - y_true # Derivata della loss rispetto alla predizione
dypred_dw2 = h # Derivata della predizione (output del hidden layer) rispetto a w2
dw2 = dloss_dypred * dypred_dw2
print(f"Gradiente rispetto a w2: {dw2}")

Gradiente rispetto a w2: -0.2062500000000001


In [8]:
#2. Gradiente rispetto a W1 (primo layer)
# dloss/dw1 = dloss/dy_pred * dy_pred/dh * dh/dw1
dypred_dh = w2 # Derivata della predizione rispetto all'output
dh_dw1 = x # Derivata dell'output del hidden layer rispetto a w1
dw1 = dloss_dypred * dypred_dh * dh_dw1
print(f"Gradiente rispetto a w1: {dw1}")


Gradiente rispetto a w1: -0.2887500000000001


# AGGIORNAMENTO PESI

In [9]:
w1_new = w1 - learning_rate * dw1
w2_new = w2 - learning_rate * dw2
print(f"Pesi aggiornati: w1={w1_new}, w2={w2_new}")


Pesi aggiornati: w1=0.528875, w2=0.720625


#Verifica del miglioramento

In [10]:
y_pred_new = (x*w1_new)*w2_new
loss_new = 0.5*(y_pred_new - y_true)**2
print(f"Nuova predizione: {y_pred_new}, Nuova Loss: {loss_new}")


Nuova predizione: 0.5716808203125, Nuova Loss: 0.026064823906586466


Ogni nodo all'interno di un grafo ha una sua memoria, il grafo memorizza gli input intermedi poichè saranno necessari per calcolare le derivate nel backforwar pass.
I grafi non sono tutti uguali:
Grafici Statisti vs Dinamici
Grafici Statici: costruiti una volta per tutte prima dell'esecuzione per ottimizzare le performance
Grafici dinamici: costriuiti al volo permettendo maggiore flessibilità durante il training
Mantenere il grafo in memoriza ha un costo. PPiù la rete è profonda più attivazioni dobbiaom salvare durante il forward pass per poterle usare nella fase di ritorno.
Il framework moderni ottimizzano il grafo eliminando operazioni inutili o combinandole per accellelare sia la predizione che l'apprendimento.

Bilaniare la velocità di calcolo con le capacità di memoria.

Immagina di voler sapere come la variazione di un variabile influenzi l'output finale. Sul grafo basta seguire tutti i sentiri possibili che collegano le due grandezze.
La regola della catena multivariabile ci insegna che dobbiamo sommare i contributi di tutti i percorsi che portano dalla variabile alla Loss finale.
La colpa dell'errore di un neruone è la somma delle colpe che vengono attribuite a tutti i neuroni precedenti.

